<a href="https://colab.research.google.com/github/code-with-Akshaya/AI-Researcher/blob/main/AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Papers to Production: Deep Research Agent**

### **Problem Statement:** Technical and engineering teams often struggle to move beyond surface-level summaries of research papers, documentation, blogs, and code repositories. Valuable insights remain locked in scattered sources, making it difficult to translate academic knowledge into production-quality decisions.


## **Download the dataset from KaggleHub**

In [ ]:
import kagglehub

# Download latest version of arXiv dataset
path = kagglehub.dataset_download("Cornell-University/arxiv")
print("Path to dataset files:", path)


Using Colab cache for faster access to the 'arxiv' dataset.
Path to dataset files: /kaggle/input/arxiv


# **Mount Google drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp -r /root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/273 /content/drive/MyDrive/arxiv_data


cp: cannot stat '/root/.cache/kagglehub/datasets/Cornell-University/arxiv/versions/273': No such file or directory


# **Data Processing**

## Load the dataset into Pandas

Norimalising the dataset which is in the form of .json into Chunks.Chunk abstracts and full text into semantic units (paragraphs, sections).
Extract metadata (authors, categories) for filtering


In [ ]:
import pandas as pd

# Load only first 100,000 rows for testing
df = pd.read_json(
    "/content/drive/MyDrive/arxiv_data/arxiv-metadata-oai-snapshot.json",
        lines=True,
            chunksize=100000
            )

            # Iterate through chunks
for chunk in df:
    print(chunk.head())
    break  # stop after first chunk




         id           submitter  \
0  704.0001      Pavel Nadolsky   
1  704.0002        Louis Theran   
2  704.0003         Hongjun Pan   
3  704.0004        David Callan   
4  704.0005  Alberto Torchinsky   

                                             authors  \
0  C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...   
1                    Ileana Streinu and Louis Theran   
2                                        Hongjun Pan   
3                                       David Callan   
4           Wael Abu-Shammala and Alberto Torchinsky   

                                               title  \
0  Calculation of prompt diphoton production cros...   
1           Sparsity-certifying Graph Decompositions   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                  comments  \
0  37 pages, 15 figures; published version   
1    To appear in Graph

In [ ]:
import pandas as pd

# Create an iterator over chunks
df_iter = pd.read_json(
    "/content/drive/MyDrive/arxiv_data/arxiv-metadata-oai-snapshot.json",
        lines=True,
            chunksize=50000   # adjust chunk size based on memory
            )

            # Get the first chunk as a DataFrame
df = next(df_iter)
print(df.head())


         id           submitter  \
0  704.0001      Pavel Nadolsky   
1  704.0002        Louis Theran   
2  704.0003         Hongjun Pan   
3  704.0004        David Callan   
4  704.0005  Alberto Torchinsky   

                                             authors  \
0  C. Bal\'azs, E. L. Berger, P. M. Nadolsky, C.-...   
1                    Ileana Streinu and Louis Theran   
2                                        Hongjun Pan   
3                                       David Callan   
4           Wael Abu-Shammala and Alberto Torchinsky   

                                               title  \
0  Calculation of prompt diphoton production cros...   
1           Sparsity-certifying Graph Decompositions   
2  The evolution of the Earth-Moon system based o...   
3  A determinant of Stirling cycle numbers counts...   
4  From dyadic $\Lambda_{\alpha}$ to $\Lambda_{\a...   

                                  comments  \
0  37 pages, 15 figures; published version   
1    To appear in Graph

### Exract the abstract safely

In [ ]:
texts = df['abstract'].dropna().tolist()
texts = [t.lower().replace("\n", " ") for t in texts]


## Embedding and Faiss Index

### Installing **FAISS** in colab

In [ ]:
!pip install faiss-cpu


### Import the **Faiss**

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss


### Embedding

In [ ]:
# Embedding model
embedder = SentenceTransformer("all-MiniLM-L6-v2")

# Create embeddings
embeddings = embedder.encode(texts, show_progress_bar=True)

# Build FAISS index
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1563 [00:00<?, ?it/s]

## Install Transformers

In [ ]:
!pip install --upgrade transformers accelerate sentencepiece



## Generate Function

### loading the Flan-T5 retrival as the workflow

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load Flan-T5 for generation
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
generator = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

# Retrieval function using arXiv dataframe
def search_with_metadata(query, k=5):
    results = []
    # Search abstracts for query
    matches = df[df['abstract'].str.contains(query, case=False, na=False)].head(k)
    for _, row in matches.iterrows():
        results.append({
           "title": row['title'],
           "authors": row['authors'],
           "year": row['update_date'],
           "abstract": row['abstract']
                 })
    return results

   # Generation function
def ask_agent_local(query, k=5):
# Retrieve top-k papers
    results = search_with_metadata(query, k)

   # Build context string
    context = "\n\n".join([
                  f"Title: {r['title']}\nAuthors: {r['authors']}\nYear: {r['year']}\nAbstract: {r['abstract']}"
                          for r in results
                              ])

    # Create prompt
    prompt = f"Question: {query}\n\nContext:\n{context}\n\nAnswer clearly and cite sources."

    # Tokenize input
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate output
    outputs = generator.generate(
            inputs["input_ids"],
            max_length=512,
            num_beams=4,
            early_stopping=True
           )

     # Decode tokens back to text
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Test the pipeline
print(ask_agent_local("Explain quantum error correction"))




Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


## extend your pipeline with Quick Mode vs Deep Mode


### Retrive Function

In [ ]:
def search_with_metadata(query, mode="quick"):
      # Encode query
     query_embedding = embedder.encode([query])

 # Quick Mode → fewer papers, faster
     if mode == "quick":
        k = 3
 # Deep Mode → more papers, richer context
     else:
        k = 10

 # Search FAISS index
     distances, indices = index.search(query_embedding, k)

 # Collect metadata for top-k results
     results = [metadata[i] for i in indices[0]]
     return results


### Generation Function with Mode

In [ ]:
def ask_agent_local(query, mode="quick"):
    results = search_with_metadata(query, mode=mode)

    context = "\n\n".join([
          f"Title: {r['title']}\nAuthors: {r['authors']}\nYear: {r['update_date']}\nAbstract: {r['abstract']}"
                for r in results
                        ])

    prompt = f"Question: {query}\n\nContext:\n{context}\n\nAnswer clearly and cite sources."

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = generator.generate(
                  inputs["input_ids"],
                  max_length=512 if mode=="quick" else 1024,  # longer answers in Deep Mode
                       num_beams=4,
                       early_stopping=True
                           )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


In [ ]:
metadata = df[['title','authors','update_date','abstract']].to_dict(orient="records")
# Quick Mode → fast, concise
print("Quick Mode:\n")
print(ask_agent_local("Explain quantum error correction", mode="quick"))

# Deep Mode → richer, more detailed
print("\nDeep Mode:\n")
print(ask_agent_local("Explain quantum error correction", mode="deep"))


Token indices sequence length is longer than the specified maximum sequence length for this model (765 > 512). Running this sequence through the model will result in indexing errors


Quick Mode:

entanglement-assisted operator quantum error correction

Deep Mode:

A formalism for quantum error correction based on operator algebras.


In [ ]:
!pip install streamlit
!streamlit run app.py


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.0 MB/s eta 0:00:00
  Attempting uninstall: cachetools
    Found existing installation: cachetools 7.0.1
    Uninstalling cachetools-7.0.1:
      Successfully uninstalled cachetools-7.0.1
Usage: streamlit run [OPTIONS] [TARGET] [ARGS]...
Try 'streamlit run --help' for help.

Error: Invalid value: File does not exist: app.py


In [ ]:
import streamlit as st

st.title("ArXiv Research Assistant")

query = st.text_input("Enter your research question:")
mode = st.radio("Choose mode:", ["quick", "deep"])

if st.button("Ask"):
   answer = ask_agent_local(query, mode=mode)
   st.write("### Answer")
   st.write(answer)

   st.write("### Retrieved Papers")
   results = search_with_metadata(query, mode=mode)
   for r in results:
       st.write(f"**{r['title']}** ({r['year']})")
       st.write(f"Authors: {r['authors']}")
       st.write(r['abstract'])
       st.write("---")


2026-02-23 11:13:07.088 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.896 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-02-23 11:13:08.910 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.923 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.932 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.951 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.963 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-23 11:13:08.973 Thread 'MainThread': mi